# Trade data quality run

We load the trades dataset and the rule file, show what we are about to validate, run the checks, and finish with timings and resource use. The data is synthetic and carries a small rate of deliberately injected issues, so most checks have something to find.

*Figures shown are indicative and depend on hardware, configuration and data.*

In [1]:
# --- configuration -------------------------------------------------
PARQUET_PATH = "./trades.parquet"   # dataset written by the generator notebook
RULES_PATH   = "./rules.json"       # rules written by the generator notebook
DUCKDB_THREADS      = 4          # None = use all cores
DUCKDB_MEMORY_LIMIT = "2GB"          # include the unit, e.g. "16GB" (a bare number is treated as GB); None = DuckDB default
SAMPLE_ROWS  = 8                    # rows shown in the preview

In [2]:
import duckdb, json, time, os, sys
import pyarrow as pa
import pyarrow.compute as pc
import numpy as np
from IPython.display import display, HTML

con = duckdb.connect()
if DUCKDB_THREADS: con.execute(f"PRAGMA threads={DUCKDB_THREADS}")
def _memlim(v):
    # accept None, '16GB', '16 GB', 16, '16' (bare numbers treated as GB)
    if v is None: return None
    s = str(v).strip().replace(' ', '')
    if not s: return None
    return s + 'GB' if s.replace('.', '', 1).isdigit() else s
_ml = _memlim(DUCKDB_MEMORY_LIMIT)
if _ml: con.execute(f"PRAGMA memory_limit='{_ml}'")

doc   = json.load(open(RULES_PATH))
RULES = doc["rules"]
con.execute(f"CREATE OR REPLACE VIEW trades AS SELECT * FROM read_parquet('{PARQUET_PATH}')")

# custom SQL functions come straight from the rule file
for m in doc["setup"]["sql_macros"]:
    con.execute(m)

# custom Python checks (vectorised, batch based, so they stay fast)
_ADDR_RE = r'^\d+\s+\S.*,\s*\S.+$'
def py_address_valid(addr):
    res = pc.match_substring_regex(addr, _ADDR_RE)
    res = pc.fill_null(res, False)
    if isinstance(res, pa.ChunkedArray):
        res = pa.concat_arrays(res.chunks) if res.num_chunks else pa.array([], type=pa.bool_())
    return res

def _np(x):
    try:    return x.to_numpy(zero_copy_only=False)
    except TypeError: return x.to_numpy()

def py_record_anomaly(risk, notional, qty):
    r = np.asarray(_np(risk),     dtype="float64")
    n = np.abs(np.asarray(_np(notional), dtype="float64"))
    q = np.asarray(_np(qty),      dtype="float64")
    anomaly = (r > 300) & np.isfinite(n) & (n > 5e7) & (q > 0)
    return pa.array(~anomaly, type=pa.bool_())

con.create_function("py_address_valid", py_address_valid, ["VARCHAR"], "BOOLEAN", type="arrow")
con.create_function("py_record_anomaly", py_record_anomaly, ["DOUBLE","DOUBLE","BIGINT"], "BOOLEAN", type="arrow")

# small helpers for the styled output
RED, BLK, INK, GREY, LT, BRD = "#EC0016", "#000000", "#1a1a1a", "#767676", "#F5F5F5", "#E0E0E0"
FONT = "font-family:Arial,Helvetica,sans-serif;"
def tile(kicker, big, note=""):
    return ('<div style="flex:1;background:'+LT+';border-top:4px solid '+RED+';padding:16px 14px;">'
            '<div style="font-size:11px;color:'+GREY+';text-transform:uppercase;letter-spacing:1.2px;font-weight:700;">'+kicker+'</div>'
            '<div style="font-size:26px;font-weight:800;color:'+RED+';margin:8px 0 4px;">'+big+'</div>'
            '<div style="font-size:12px;color:'+INK+';">'+note+'</div></div>')
def panel(inner):
    return '<div style="max-width:960px;'+FONT+'color:'+INK+';background:#fff;padding:4px 2px;">'+inner+'</div>'
print("loaded", len(RULES), "rules |", doc["dataset"], "|", PARQUET_PATH)

loaded 54 rules | trades | ./trades.parquet


In [3]:
# ------- what we are about to validate: the dataset -------
n_rows = con.execute("SELECT count(*) FROM trades").fetchone()[0]
schema = con.execute("PRAGMA table_info('trades')").fetchall()
size_gb = os.path.getsize(PARQUET_PATH) / 1e9
samp = con.execute(f"SELECT * FROM trades LIMIT {SAMPLE_ROWS}").fetchdf()

def fmt(v):
    s = str(v)
    return s[:22] + "…" if len(s) > 23 else s
head = "".join('<th style="padding:6px 9px;background:'+BLK+';color:#fff;font-size:11px;text-align:left;white-space:nowrap;">'+c+"</th>" for c in samp.columns)
rows = ""
for _, r in samp.iterrows():
    tds = "".join('<td style="padding:5px 9px;border-bottom:1px solid '+BRD+';font-size:11.5px;white-space:nowrap;">'+fmt(v)+"</td>" for v in r)
    rows += "<tr>"+tds+"</tr>"
chips = "".join('<span style="display:inline-block;background:'+LT+';border:1px solid '+BRD+';padding:3px 8px;margin:3px;font-size:11px;">'
                +name+' <span style="color:'+GREY+';">'+typ+"</span></span>" for _,name,typ,*_ in schema)
html = (
 '<div style="font-size:24px;font-weight:800;border-top:4px solid '+RED+';padding-top:12px;">The dataset we validate</div>'
 '<div style="display:flex;gap:14px;margin:16px 0;">'
 + tile("Rows", f"{n_rows:,}")
 + tile("Columns", str(len(schema)))
 + tile("Size on disk", f"{size_gb:,.2f} GB", "Parquet")
 + '</div>'
 '<div style="overflow-x:auto;border:1px solid '+BRD+';"><table style="border-collapse:collapse;">'
 '<tr>'+head+'</tr>'+rows+'</table></div>'
 '<div style="font-size:11px;color:'+GREY+';margin:6px 0 10px;">First '+str(SAMPLE_ROWS)+' rows, values shortened for display.</div>'
 '<div style="font-size:13px;font-weight:700;margin-top:8px;">All columns</div>'
 '<div style="margin-top:6px;">'+chips+'</div>'
)
display(HTML(panel(html)))

trade_id,client_id,parent_order_id,trade_ref,isin,ticker,side,currency,settlement_ccy,venue,status,asset_class,quantity,price,gross_amount,fees,tax,net_amount,notional,trade_date,settlement_date,booked_ts,trader_lei,counterparty,risk_score,metadata,tags,is_cancelled,book_geo,region,sub_region,account_no,address,qty_str,date_str,account_age_yrs,notes,cash_flow
0,18257,TRD-00000000,TRD-00000000,GB0000000000,SAP,BUY,JPY,JPY,XETR,SETTLED,FX,75220,686.1634,51613210.95,198.43,106.3,51612906.22,51613210.95,2025-03-11 00:00:00,2025-03-12 00:00:00,2026-07-06 04:06:57.63…,LEI00000000000000000,MORGAN STANLEY,98.64,"{""source_system"":""MURE…",['EQUITY' 'LDN'],False,"51.5877,0.1687",EMEA,LONDON,4001919257537193,"46 King St, LONDON",75220,2025-03-11,12,Reviewed VERIFIED,-51612906.22
1,32005,TRD-00000001,TRD-00000001,GB0763583521,SAP,BUY,GBP,GBP,XLON,NEW,FX,39535,531.7964,21024570.67,13.54,43.55,21024513.58,21024570.67,2025-05-30 00:00:00,2025-06-02 00:00:00,2026-07-05 16:06:57.63…,LEI17996019076358352,MORGAN STANLEY,97.11,"{""source_system"":""CALY…",['EQUITY' 'TOK'],False,"51.6521,0.1919",APAC,TOKYO,4916338506082832,"154 King St, LONDON",39535,2025-05-30,6,Amended VERIFIED,-21024513.58
2,48851,TRD-00000002,TRD-00000002,GB9175788342,AAPL,SELL,XXX,XXX,XPAR,SETTLED,FX,72483,350.0252,25370876.57,89.34,29.67,25370757.56,25370876.57,2025-08-08 00:00:00,NaT,2026-07-06 11:06:57.63…,LEI60787363917578834,MORGAN STANLEY,72.23,"{""source_system"":""CALY…",['EQUITY' 'TOK'],False,"80.0,80.0",APAC,TOKYO,5500005555555559,"68 High Rd, LONDON",N/A,2025-08-08,28,Reviewed VERIFIED,25370757.56
3,53187,TRD-00000003,TRD-00000003,GB4785189823,BP,BUY,USD,USD,XNYS,SETTLED,EQUITY,11479,330.5791,3794717.49,186.56,42.6,3794488.33,3794717.49,2025-02-04 00:00:00,2025-02-07 00:00:00,2026-07-06 02:06:57.63…,LEI31803788478518982,MORGAN STANLEY,19.99,"{""source_system"":""SUMM…",['EQUITY' 'NY'],False,"51.5902,0.1054",EMEA,LONDON,4539578763621486,"133 King St, LONDON",11479,2025-02-04,3,Reviewed VERIFIED,-3794488.33
4,10538,TRD-00000004,TRD-00000004,GB0511910364,TSLA,BUY,EUR,EUR,XTKS,SETTLED,EQUITY,43553,994.6198,43318676.15,20.25,126.66,43318529.24,43318676.15,2025-01-07 00:00:00,2025-01-09 00:00:00,2026-07-06 06:06:57.63…,LEI35942711051191036,MORGAN STANLEY,99.31,"{""source_system"":""MURE…",['EQUITY' 'NY'],False,"51.6405,0.2729",APAC,TOKYO,4539578763621486,"156 Bank Ave, LONDON",43553,2025-01-07,30,Reviewed VERIFIED,-43318529.24
5,5825,TRD-00000005,TRD-00000005,GB2964206925,BP,SELL,USD,USD,XETR,SETTLED,FX,82478,472.8034,38995878.83,97.48,17.59,38995763.76,38995878.83,2025-05-14 00:00:00,2025-05-15 00:00:00,2026-07-05 23:06:57.63…,LEI44145009296420692,GOLDMAN SACH,56.55,"{""source_system"":""MURE…",['EQUITY' 'NY'],False,"51.5504,0.1286",EMEA,LONDON,5500005555555559,"109 King St, LONDON",82478,2025-05-14,35,Reviewed VERIFIED,38995763.76
6,48986,TRD-00000006,TRD-00000006,GB8616781376,TSLA,BUY,USD,USD,XLON,SETTLED,EQUITY,66438,2.4918,165550.21,39.14,133.68,165377.39,165550.21,2025-07-01 00:00:00,2025-07-04 00:00:00,2026-07-05 01:06:57.63…,LEI88402906861678137,BARCLAYS,66.07,"{""source_system"":""MURE…",[],False,"51.6474,0.1785",AMER,NEW YORK,4001919257537193,"119 High Rd, LONDON",66438,2025-07-01,27,Amended VERIFIED,-165377.39
7,24735,TRD-00000007,TRD-00000007,GB7065631467,TSLA,SELL,CAD,CAD,XTKS,SETTLED,FX,34774,992.0634,34498012.67,137.23,39.3,34497836.14,34498012.67,2025-12-16 00:00:00,2025-12-18 00:00:00,2026-07-05 20:06:57.63…,LEI36873150706563146,GOLDMAN SACHS,57.21,"{""source_system"":""MURE…",['FX'],False,"51.698,0.1606",EMEA,LONDON,4916338506082832,"158 High Rd, LONDON",N/A,2025-12-16,-3,Order VERIFIED,34497836.14


In [4]:
# ------- what we are about to validate: the rules -------
DIM_ORDER = ["Dataset","Schema","Completeness","Validity","Consistency","Timeliness","Uniqueness"]
DIM_Q = {"Dataset":"Did the data arrive whole?","Schema":"Is it shaped right?",
 "Completeness":"Is the data there?","Validity":"Does it look right?",
 "Consistency":"Do the fields agree?","Timeliness":"Is it fresh?","Uniqueness":"Are we double-counting?"}
groups = {d:[r for r in RULES if r["dimension"]==d] for d in DIM_ORDER}
blocks = ""
for d in DIM_ORDER:
    rs = groups[d]
    if not rs: continue
    items = ""
    for r in rs:
        expr = r.get("sql","") or (r.get("variant","")+" duplicate check")
        expr = expr.replace("{src}","trades")
        expr_disp = (expr[:96]+"…") if len(expr)>97 else expr
        badge = ""
        if r.get("requires"):
            kind = "custom Python" if any(x.startswith("py_") for x in r["requires"]) else "custom SQL"
            badge = ' <span style="background:'+RED+';color:#fff;font-size:9.5px;padding:1px 6px;letter-spacing:.5px;">'+kind.upper()+'</span>'
        items += ('<div style="padding:7px 0;border-bottom:1px solid '+BRD+';">'
                  '<div style="font-size:13px;">'+r["business_label"]+badge+'</div>'
                  '<div style="font-size:10.5px;color:'+GREY+';font-family:Consolas,Menlo,monospace;margin-top:2px;">'+expr_disp+'</div></div>')
    blocks += ('<div style="border:1px solid '+BRD+';border-top:4px solid '+RED+';padding:12px 14px;margin-top:14px;">'
               '<div style="display:flex;justify-content:space-between;align-items:baseline;">'
               '<div style="font-size:15px;font-weight:800;">'+d+' <span style="color:'+GREY+';font-weight:400;font-size:12px;">('+str(len(rs))+')</span></div>'
               '<div style="font-size:11.5px;color:'+RED+';font-weight:700;">'+DIM_Q[d]+'</div></div>'+items+'</div>')
html = ('<div style="font-size:24px;font-weight:800;border-top:4px solid '+RED+';padding-top:12px;">The checks we run</div>'
        '<div style="font-size:13px;color:#3a3a3a;margin-top:6px;">'+str(len(RULES))+' checks across the standard quality dimensions. '
        'Each shows its plain meaning and, underneath, the expression the engine runs, shortened for display.</div>'+blocks)
display(HTML(panel(html)))

In [5]:
# ------- run the checks (the notebook plans its own passes) -------
SRC = "trades"
def sub(s): return s.replace("{src}", SRC)

row_all  = [r for r in RULES if r["result_kind"]=="row"]
row_py   = [r for r in row_all if any(x.startswith("py_") for x in r.get("requires",[]))]
row_sql  = [r for r in row_all if r not in row_py]
aggs     = [r for r in RULES if r["result_kind"]=="aggregate"]
dedups   = [r for r in RULES if r["result_kind"]=="dedup"]
ds_rule  = next(r for r in RULES if r["result_kind"]=="dataset")
sch_f    = next(r for r in RULES if r["result_kind"]=="schema_fields")
sch_t    = next(r for r in RULES if r["result_kind"]=="schema_types")

results, timings = {}, {}

# pass 0: structure (metadata only)
t = time.perf_counter()
ok = con.execute(f"SELECT ({sub(ds_rule['sql'])}) FROM {SRC}").fetchone()[0]
results[ds_rule["id"]] = {"status": "PASS" if ok else "FLAG", "detail": f"{n_rows:,} rows"}
actual_cols = [r[1] for r in schema]; actual_types = {r[1]: r[2] for r in schema}
missing = [c for c in sch_f["expected"] if c not in actual_cols]
extra   = [c for c in actual_cols if c not in sch_f["expected"]]
results[sch_f["id"]] = {"status": "PASS" if not missing and not extra else "FLAG",
                        "detail": f"{len(actual_cols)}/{len(sch_f['expected'])} columns present"}
bad = {c: (t_, actual_types.get(c)) for c, t_ in sch_t["expected"].items() if actual_types.get(c) != t_}
results[sch_t["id"]] = {"status": "PASS" if not bad else "FLAG",
                        "detail": "types match" if not bad else f"mismatch: {bad}"}
timings["structure"] = time.perf_counter() - t
print(f"structure checks          {timings['structure']:>8.2f}s")

# pass 1: every SQL row rule in one combined scan
t = time.perf_counter()
sel = ", ".join(f'count_if(NOT ({sub(r["sql"])})) AS c{i}' for i, r in enumerate(row_sql))
vals = con.execute(f"SELECT {sel} FROM {SRC}").fetchone()
for r, v in zip(row_sql, vals):
    results[r["id"]] = {"status": "PASS" if v == 0 else "FLAG", "rows": v}
timings["sql row scan"] = time.perf_counter() - t
print(f"sql row scan ({len(row_sql)} rules)   {timings['sql row scan']:>8.2f}s")

# pass 2: the custom Python checks
t = time.perf_counter()
sel = ", ".join(f'count_if(NOT ({sub(r["sql"])})) AS p{i}' for i, r in enumerate(row_py))
vals = con.execute(f"SELECT {sel} FROM {SRC}").fetchone()
for r, v in zip(row_py, vals):
    results[r["id"]] = {"status": "PASS" if v == 0 else "FLAG", "rows": v}
timings["python checks"] = time.perf_counter() - t
print(f"python checks ({len(row_py)} rules)   {timings['python checks']:>8.2f}s")

# pass 3: aggregate checks in one statement
t = time.perf_counter()
sel = ", ".join(f'({sub(r["sql"])}) AS a{i}' for i, r in enumerate(aggs))
vals = con.execute(f"SELECT {sel} FROM {SRC}").fetchone()
for r, v in zip(aggs, vals):
    results[r["id"]] = {"status": "PASS" if v else "FLAG", "detail": "within band" if v else "outside band"}
timings["aggregates"] = time.perf_counter() - t
print(f"aggregate checks ({len(aggs)})    {timings['aggregates']:>8.2f}s")

# pass 4: duplicate checks
t = time.perf_counter()
parts, order = [], []
for i, r in enumerate(dedups):
    cols, exact = r["columns"], r.get("variant") == "exact"
    if exact:
        key = ", ".join(cols)
        parts.append(f"(SELECT count(*) FROM {SRC}) - (SELECT count(*) FROM (SELECT DISTINCT {key} FROM {SRC}))")
    else:
        expr = cols[0] if len(cols) == 1 else "concat_ws('|', " + ", ".join(c+"::VARCHAR" for c in cols) + ")"
        parts.append(f"(SELECT count(*) FROM {SRC}) - (SELECT approx_count_distinct({expr}) FROM {SRC})")
    order.append(r)
vals = con.execute("SELECT " + ", ".join(f"({p}) AS d{i}" for i, p in enumerate(parts))).fetchone()
for r, v in zip(order, vals):
    v = max(v, 0)
    if r.get("variant") == "approx":
        heavy = v > 0.25 * n_rows
        results[r["id"]] = {"status": "FLAG" if heavy else "PASS", "screen": v}
    else:
        results[r["id"]] = {"status": "PASS" if v == 0 else "FLAG", "rows": v}
timings["duplicates"] = time.perf_counter() - t
print(f"duplicate checks ({len(dedups)})    {timings['duplicates']:>8.2f}s")

total_s = sum(timings.values())
flagged = sum(1 for v in results.values() if v["status"] == "FLAG")
print("-" * 44)
print(f"{len(RULES)} checks on {n_rows:,} rows in {total_s:,.2f}s  |  {flagged} checks flagged rows")

structure checks              0.08s
sql row scan (38 rules)     139.53s
python checks (2 rules)      27.43s
aggregate checks (7)        3.11s
duplicate checks (4)       14.69s
--------------------------------------------
54 checks on 50,000,000 rows in 184.83s  |  35 checks flagged rows


In [6]:
# ------- results, by dimension -------
def badge(st):
    if st == "PASS":
        return '<span style="background:#e9e9e9;color:#111;font-size:10.5px;font-weight:700;padding:2px 10px;">PASS</span>'
    return '<span style="background:'+RED+';color:#fff;font-size:10.5px;font-weight:700;padding:2px 10px;">FLAGGED</span>'
def pct(v):
    if n_rows == 0: return ""
    p = 100.0 * v / n_rows
    return "<0.01%" if 0 < p < 0.01 else f"{p:,.2f}%"
blocks = ""
for d in DIM_ORDER:
    rs = groups[d]
    if not rs: continue
    items = ""
    for r in rs:
        res = results[r["id"]]
        right = res.get("detail", "")
        if "screen" in res:
            right = ('no heavy duplication (fast screen, coarse resolution)' if res["status"] == "PASS"
                     else '<b style="color:'+RED+';">&asymp;{:,} rows</b> ({}) fast estimate'.format(res["screen"], pct(res["screen"])))
        elif "rows" in res:
            right = ("no rows flagged" if res["rows"] == 0
                     else f'<b style="color:'+RED+';">{:,} rows</b> ({})'.format(res["rows"], pct(res["rows"])))
        items += ('<div style="display:flex;justify-content:space-between;gap:10px;padding:7px 0;border-bottom:1px solid '+BRD+';">'
                  '<div style="font-size:13px;">'+r["business_label"]+'</div>'
                  '<div style="font-size:12px;white-space:nowrap;">'+right+' &nbsp;'+badge(res["status"])+'</div></div>')
    nflag = sum(1 for r in rs if results[r["id"]]["status"] == "FLAG")
    blocks += ('<div style="border:1px solid '+BRD+';border-top:4px solid '+RED+';padding:12px 14px;margin-top:14px;">'
               '<div style="font-size:15px;font-weight:800;">'+d+
               ' <span style="color:'+GREY+';font-weight:400;font-size:12px;">('+str(len(rs))+' checks, '+str(nflag)+' flagged)</span></div>'
               + items + '</div>')
html = ('<div style="font-size:24px;font-weight:800;border-top:4px solid '+RED+';padding-top:12px;">What the run found</div>'
        '<div style="font-size:12.5px;color:'+GREY+';margin-top:6px;">The data carries a small rate of injected issues by design, '
        'so flagged rows here show the checks working, not a broken feed.</div>' + blocks)
display(HTML(panel(html)))

In [7]:
# ------- timing, throughput and resources -------
# peak process memory, cross platform (psutil if present, else OS API, else n/a)
def peak_memory_gb():
    try:
        import psutil
        mi = psutil.Process().memory_info()
        return (getattr(mi, "peak_wset", None) or mi.rss) / 1e9
    except Exception:
        pass
    if sys.platform.startswith("win"):
        try:
            import ctypes, ctypes.wintypes as wt
            class PMC(ctypes.Structure):
                _fields_ = [("cb", wt.DWORD), ("PageFaultCount", wt.DWORD),
                            ("PeakWorkingSetSize", ctypes.c_size_t), ("WorkingSetSize", ctypes.c_size_t),
                            ("QuotaPeakPagedPoolUsage", ctypes.c_size_t), ("QuotaPagedPoolUsage", ctypes.c_size_t),
                            ("QuotaPeakNonPagedPoolUsage", ctypes.c_size_t), ("QuotaNonPagedPoolUsage", ctypes.c_size_t),
                            ("PagefileUsage", ctypes.c_size_t), ("PeakPagefileUsage", ctypes.c_size_t)]
            pmc = PMC(); pmc.cb = ctypes.sizeof(PMC)
            ctypes.windll.psapi.GetProcessMemoryInfo(ctypes.windll.kernel32.GetCurrentProcess(),
                                                     ctypes.byref(pmc), pmc.cb)
            return pmc.PeakWorkingSetSize / 1e9
        except Exception:
            return None
    try:
        import resource
        ru = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return ru / 1e6 if sys.platform.startswith("linux") else ru / 1e9
    except Exception:
        return None

threads = con.execute("SELECT current_setting('threads')").fetchone()[0]
memlim  = con.execute("SELECT current_setting('memory_limit')").fetchone()[0]
peak_gb = peak_memory_gb()
rps  = n_rows / total_s if total_s else 0
rpsc = rps / int(threads) if int(threads) else rps
bars, mx = "", max(timings.values())
for name, s in timings.items():
    w = max(2, int(100 * s / mx))
    bars += ('<div style="display:flex;align-items:center;margin-bottom:8px;">'
             '<div style="width:170px;font-size:12.5px;">'+name+'</div>'
             '<div style="flex:1;background:#ededed;height:20px;"><div style="width:'+str(w)+'%;background:'+RED+';height:20px;"></div></div>'
             '<div style="width:80px;text-align:right;font-size:12.5px;">'+f"{s:,.2f}s"+'</div></div>')
html = ('<div style="font-size:24px;font-weight:800;border-top:4px solid '+RED+';padding-top:12px;">How fast, on what</div>'
 '<div style="display:flex;gap:14px;margin:16px 0;">'
 + tile("Rows validated", f"{n_rows:,}")
 + tile("Checks run", str(len(RULES)))
 + tile("Total time", f"{total_s:,.1f}s")
 + tile("Throughput", f"{rps:,.0f}", "rows per second, all checks")
 + '</div>'
 '<div style="border:1px solid '+BRD+';padding:14px 16px;">'
 '<div style="font-size:12px;font-weight:800;text-transform:uppercase;letter-spacing:1.2px;margin-bottom:10px;">Time by phase</div>'
 + bars + '</div>'
 '<div style="font-size:12.5px;color:'+INK+';margin-top:12px;">Ran on <b>'+str(threads)+' threads</b>, DuckDB memory limit <b>'+str(memlim)+'</b>, '
 'peak process memory <b>'+(f"{peak_gb:,.1f} GB" if peak_gb else "n/a")+'</b>, one machine, one process. '
 'About '+f"{rpsc:,.0f}"+' rows per second per thread.</div>'
 '<div style="font-size:10.5px;color:'+GREY+';margin-top:10px;border-top:1px solid '+BRD+';padding-top:8px;">'
 'Indicative figures. Timings depend on hardware, configuration and data; the Python checks trade some speed for the ability to run our own code.</div>')
display(HTML(panel(html)))